In [1]:
import json
import math
from pathlib import Path
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

In [2]:
DOCS_DIR        = "../../data/parsed_json/"
CHROMA_DIR      = "../../data/chroma_db/"
COLLECTION_NAME = "rfp_docs"
MAX_CHUNK_SIZE  = 1000
BATCH_SIZE      = 500

# 임베딩 모델 선택
# "text-embedding-3-small" : 베이스라인, OpenAI API
# "text-embedding-3-large" : 고성능, OpenAI API
# "bge-m3"                 : 한국어 강점, 로컬 무료
# "ko-sroberta-multitask"  : 한국어 특화, 로컬 무료
# "embed-multilingual-v3"  : Cohere, 한국어 포함 100개 언어
EMBEDDING_MODEL = "bge-m3"

1. 유틸리티

In [3]:
def clean(val):
    # NaN / None → 빈 문자열 (Chroma 메타데이터는 NaN·None 불가)
    if val is None:
        return ""
    if isinstance(val, float) and math.isnan(val):
        return ""
    return val

In [4]:
def build_payload(doc: dict, section: dict, block: dict) -> dict:
    # 청크 메타데이터(페이로드) 생성
    meta = doc.get("metadata", {})
    return {
        "doc_id":        str(clean(doc.get("doc_id"))),
        "file_name":     str(clean(doc.get("file_name"))),
        "source_format": str(clean(doc.get("source_format"))),
        "사업명":         str(clean(meta.get("사업명"))),
        "발주기관":       str(clean(meta.get("발주기관"))),
        "사업유형":       str(clean(meta.get("사업유형"))),
        "사업금액":       float(clean(meta.get("사업금액")) or 0.0),
        "공고번호":       str(clean(meta.get("공고번호"))),
        "공고차수":       float(clean(meta.get("공고차수")) or 0.0),
        "공개일자":       str(clean(meta.get("공개일자"))),
        "입찰참여시작일":  str(clean(meta.get("입찰참여시작일"))),
        "입찰참여마감일":  str(clean(meta.get("입찰참여마감일"))),
        "재공고여부":     bool(meta.get("재공고여부", False)),
        "linked_doc_id": str(clean(meta.get("linked_doc_id"))),
        "사업요약":       str(clean(meta.get("사업요약"))),
        "header_path":   " > ".join(section.get("header_path", [])),
        "section_id":    str(clean(section.get("section_id"))),
        "block_id":      str(clean(block.get("block_id"))),
        "block_type":    str(clean(block.get("type"))),
        "table_type":    str(clean(block.get("table_type"))),
    }

In [5]:
#  Chunk dataclass
from dataclasses import dataclass, field

@dataclass
class Chunk:
    content: str
    doc_id: str
    block_id: str
    header_path: str
    metadata: dict = field(default_factory=dict)

2. 청킹

In [6]:
def chunk_section(section: dict) -> list[dict]:

    header_prefix = " > ".join(section.get("header_path", []))
    results = []
    current_text = ""
    last_text_block = None

    for block in section.get("blocks", []):
        if block.get("is_decorative"):
            continue
        if block["type"] == "table":
            # 쌓인 텍스트 먼저 방출
            if current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = ""
                last_text_block = None
            # 표 단독 방출
            results.append({
                "content": f"[섹션: {header_prefix}]\n\n{block['content']}",
                "block":   block,
            })
        else:
            # 텍스트 블록 누적
            if len(current_text) + len(block["content"]) > MAX_CHUNK_SIZE and current_text.strip():
                results.append({
                    "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
                    "block":   last_text_block,
                })
                current_text = block["content"] + "\n\n"
            else:
                current_text += block["content"] + "\n\n"
            last_text_block = block

    # 남은 텍스트 방출
    if current_text.strip():
        results.append({
            "content": f"[섹션: {header_prefix}]\n\n{current_text.strip()}",
            "block":   last_text_block,
        })

    return results

In [7]:
def process_doc(doc: dict) -> tuple[list[str], list[dict]]:
    # JSON 문서 1개 → (청크 텍스트 리스트, 메타데이터 리스트)
    texts, metas = [], []

    # extraction_warnings 있으면 경고 출력 후 계속 처리
    warnings = doc.get("qa", {}).get("extraction_warnings", [])
    if warnings:
        print(f"  [WARN] {doc['doc_id']} — extraction_warnings: {warnings}")

    meta = doc.get("metadata", {})
    summary = str(clean(meta.get("사업요약")))
    사업명 = str(clean(meta.get("사업명")))
    발주기관 = str(clean(meta.get("발주기관")))

    for section in doc.get("sections", []):
        chunks = chunk_section(section)
        for item in chunks:
            # 사업명·발주기관·요약을 앞에 붙여 retrieval 정확도 향상
            prefix = (
                f"[사업명] {사업명}\n"
                f"[발주기관] {발주기관}\n"
                f"[요약] {summary}\n\n"
            )
            texts.append(prefix + item["content"])
            metas.append(build_payload(doc, section, item["block"] or {}))
    return texts, metas

3. 임베딩 모델

In [8]:
def load_embedding_model(name: str):
    if name == "text-embedding-3-small":
        return OpenAIEmbeddings(model="text-embedding-3-small")
    elif name == "text-embedding-3-large":
        return OpenAIEmbeddings(model="text-embedding-3-large")
    elif name == "bge-m3":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="BAAI/bge-m3",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "ko-sroberta-multitask":
        from langchain_huggingface import HuggingFaceEmbeddings
        return HuggingFaceEmbeddings(
            model_name="snunlp/KR-SBERT-V40K-klueNLI-augSTS",
            model_kwargs={"device": "cuda"},
            encode_kwargs={"normalize_embeddings": True},
        )
    elif name == "embed-multilingual-v3":
        from langchain_cohere import CohereEmbeddings
        return CohereEmbeddings(model="embed-multilingual-v3.0")
    else:
        raise ValueError(f"알 수 없는 임베딩 모델: {name}")


4. Vector DB / Retrieval

In [9]:
def get_vectorstore(embedding_model=None) -> Chroma:
    # 저장된 Chroma DB 불러오기 (검색 전용)
    if embedding_model is None:
        embedding_model = load_embedding_model(EMBEDDING_MODEL)
    return Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_model,
        persist_directory=CHROMA_DIR,
    )

In [10]:
def get_retriever(vectorstore: Chroma, search_type: str = "similarity", k: int = 5):

    if search_type == "mmr":
        return vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={"k": k, "fetch_k": k * 4, "lambda_mult": 0.5},
        )
    return vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k},
    )

In [11]:
def save_vectorstore():
    docs_dir = Path(DOCS_DIR)
    json_files = sorted(docs_dir.glob("*.json"))

    all_texts, all_metas = [], []

    for json_file in json_files:
        with open(json_file, encoding="utf-8") as f:
            doc = json.load(f)

        texts, metas = process_doc(doc)
        all_texts.extend(texts)
        all_metas.extend(metas)

    embedding_model = load_embedding_model(EMBEDDING_MODEL)
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embedding_model,
        persist_directory=CHROMA_DIR,
    )

    for start in range(0, len(all_texts), BATCH_SIZE):
        end = start + BATCH_SIZE
        vectorstore.add_texts(
            texts=all_texts[start:end],
            metadatas=all_metas[start:end],
        )
        print(f"  저장 완료: {min(end, len(all_texts))}/{len(all_texts)}")

    print(f"\nChroma 저장 완료 (총 {vectorstore._collection.count()}개 청크)")

5. 실행

In [13]:
# save_vectorstore() 호출
save_vectorstore()

  [WARN] D002 — extraction_warnings: ['빈 표 감지 (blk#1)', 'high_decorative_table_ratio: 23.7%']
  [WARN] D003 — extraction_warnings: ['빈 표 감지 (blk#50)', '빈 표 감지 (blk#300)', 'high_decorative_table_ratio: 19.7%']
  [WARN] D004 — extraction_warnings: ['high_decorative_table_ratio: 16.5%']
  [WARN] D005 — extraction_warnings: ['빈 표 감지 (blk#1)', '빈 표 감지 (blk#101)', '빈 표 감지 (blk#315)', 'high_decorative_table_ratio: 26.4%']
  [WARN] D006 — extraction_warnings: ['high_decorative_table_ratio: 20.3%']
  [WARN] D007 — extraction_warnings: ['빈 표 감지 (blk#277)', '빈 표 감지 (blk#312)', '빈 표 감지 (blk#362)', '빈 표 감지 (blk#367)', '빈 표 감지 (blk#368)']
  [WARN] D009 — extraction_warnings: ['빈 표 감지 (blk#229)', '빈 표 감지 (blk#229)', '빈 표 감지 (blk#229)', 'high_decorative_table_ratio: 28.7%']
  [WARN] D010 — extraction_warnings: ['빈 표 감지 (blk#6)', 'high_decorative_table_ratio: 47.8%']
  [WARN] D011 — extraction_warnings: ['빈 표 감지 (blk#99)']
  [WARN] D012 — extraction_warnings: ['빈 표 감지 (blk#30)', '빈 표 감지 (blk#36)', '빈 표

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

  저장 완료: 500/17382
  저장 완료: 1000/17382
  저장 완료: 1500/17382
  저장 완료: 2000/17382
  저장 완료: 2500/17382
  저장 완료: 3000/17382
  저장 완료: 3500/17382
  저장 완료: 4000/17382
  저장 완료: 4500/17382
  저장 완료: 5000/17382
  저장 완료: 5500/17382
  저장 완료: 6000/17382
  저장 완료: 6500/17382
  저장 완료: 7000/17382
  저장 완료: 7500/17382
  저장 완료: 8000/17382
  저장 완료: 8500/17382
  저장 완료: 9000/17382
  저장 완료: 9500/17382
  저장 완료: 10000/17382
  저장 완료: 10500/17382
  저장 완료: 11000/17382
  저장 완료: 11500/17382
  저장 완료: 12000/17382
  저장 완료: 12500/17382
  저장 완료: 13000/17382
  저장 완료: 13500/17382
  저장 완료: 14000/17382
  저장 완료: 14500/17382
  저장 완료: 15000/17382
  저장 완료: 15500/17382
  저장 완료: 16000/17382
  저장 완료: 16500/17382
  저장 완료: 17000/17382
  저장 완료: 17382/17382

Chroma 저장 완료 (총 17382개 청크)


6. 검색 테스트

In [ ]:
vs = get_vectorstore()
retriever = get_retriever(vs, k=5)

query = "보안 요구사항이 있는 사업은?"
results = retriever.invoke(query)

for i, doc in enumerate(results, 1):
    print(f"[{i}] 사업명: {doc.metadata.get('사업명')}")
    print(f"     발주기관: {doc.metadata.get('발주기관')}")
    print(f"     섹션: {doc.metadata.get('header_path')}")
    print(f"     내용:\n{doc.page_content[:300]}")
    print()

Loading weights: 100%
 391/391 [00:00<00:00, 11545.04it/s]
[1] 사업명: 경영정보시스템 기능개선
     발주기관: 부산관광공사
     섹션: (서두) > 2. 요구사항 세부내용
     내용:
[사업명] 경영정보시스템 기능개선
[발주기관] 부산관광공사
[요약] - 사업개요: 부산관광공사 경영정보시스템 기능개선 용역
- 추진배경: 업무환경 변화에 따른 기능개선 필요
- 사업범위: 경영정보시스템 기능 개선 및 최적화
- 기대효과: 업무처리 효율성 및 편의성 향상
- 추진목표: 사용자 요구사항 반영 및 업무환경 개선

[섹션: (서두) > 2. 요구사항 세부내용]

| 요구사항 분류 | 보안 요구사항 |
| --- | --- |
| 요구사항 번호 | SER-001 |
| 요구사항 명칭 | 보안사항 |
| 요구사항상세설명 | 정

[2] 사업명: [재공고]차세대 통합정보시스템(ERP) 구축
     발주기관: 한국가스공사
     섹션: Ⅲ. 일반사항 > 1. 상세 요구사항 > 11) 보안 요구사항
     내용:
[사업명] [재공고]차세대 통합정보시스템(ERP) 구축
[발주기관] 한국가스공사
[요약] - 사업목적: 기술지원 종료에 대비한 ERP 업그레이드로 안정적 서비스 제공 및 업무생산성 향상
- 주요 사업내용: SAP ERP 전환, 업무 프로세스 개선, 생산공급 시스템 재구축, 전사 포털 및 업무 포털 신규 구축, 업무 프로세스 자산화 및 관리시스템 도입
- 신규 도입 소프트웨어 내역: Process Asset Management System(PAMS) 솔루션, EAI/API 연계 솔루션, SSO 통합 인증 솔루션, 웹전환 솔루션, SA

[3] 사업명: 경영정보시스템 기능개선
     발주기관: 부산관광공사
     섹션: (서두) > 2. 요구사항 세부내용
     내용:
[사업명] 경영정보시스템 기능개선
[발주기관] 부산관광공사
[요약] - 사업개요: 부산관광공사 경영정보시스템 기능개선 용역
- 추진배경: 업무환경 변화에 따른 기능개선 필요
- 사업범위: 경영정보시스템 기능 개선 및 최적화
- 기대효과: 업무처리 효율성 및 편의성 향상
- 추진목표: 사용자 요구사항 반영 및 업무환경 개선

[섹션: (서두) > 2. 요구사항 세부내용]

| 요구사항 분류 | 보안 요구사항 |
| --- | --- |
| 요구사항 번호 | SER-004 |
| 요구사항 명칭 | 사업수행 인력 보안 |
| 요구사항상세

[4] 사업명: 2024년도 GKL  그룹웨어 시스템 구축 용역
     발주기관: 그랜드코리아레저(주)
     섹션: Ⅵ. 제안 안내사항 127 > 4. 상세 요구사항 > 7) 보안 요구사항
     내용:
[사업명] 2024년도 GKL  그룹웨어 시스템 구축 용역
[발주기관] 그랜드코리아레저(주)
[요약] - 사업개요: GKL 그룹웨어 시스템 구축사업
- 추진배경: 그룹웨어 및 기록물관리, 사내SNS 시스템 노후화로 인한 개선 필요, 정부권장정책 적용, 웹기반 시스템 필요
- 사업범위: 그룹웨어 시스템 구축, 기록물관리 시스템 구축, 업무용 메신저 시스템 구축
- 기대효과: 시스템 안정성 확보, 관리 효율성 향상, 사용자 편의 증대

[섹션: Ⅵ. 제안 안내사항 127 > 4. 상세 요구사항 > 7) 보안 요구사항]

| 요구사항 분

[5] 사업명: 실손보험 청구 전산화 시스템 구축 사업
     발주기관: 사단법인 보험개발원
     섹션: (서두) > 2. 상세 요구사항 > 9) 보안 요구사항(Security Requirement)
     내용:
[사업명] 실손보험 청구 전산화 시스템 구축 사업
[발주기관] 사단법인 보험개발원
[요약] - 사업 개요: 실손보험 청구 전산화 시스템 구축
- 추진배경: 보험금 청구 과정의 불편과 비효율성 개선 필요성
- 사업범위: 전국 요양기관과 보험사 연계 시스템 구축
- 기대효과: 보험가입자의 편의 증대, 종이서류 사용 절감, 청구율 증가
- 추진목표: 실손보험 청구 과정의 전산화를 통한 효율적인 보험금 지급 프로세스 구축

[섹션: (서두) > 2. 상세 요구사항 > 9) 보안 요구사항(Security Requirement)]

| 요구사

7. 임베딩 모델 비교